<a href="https://colab.research.google.com/github/Shaxzod1991/IT_project_TR_TP/blob/main/%D0%A2%D0%B0%D1%81%D0%BA%D0%BE_%D0%94%D0%97_%D0%9A%D0%97_%D0%B4%D0%BD%D0%B8_%D0%BF%D1%80%D0%BE%D1%81%D1%80%D0%BE%D1%87%D0%BA%D0%B8_26_05_2026.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
import os
import re
from openpyxl.styles import Alignment, Border, Font, PatternFill, Side
from openpyxl.utils import get_column_letter

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# ***Пути к исходникам***

In [2]:
path_input = r'/content/drive/MyDrive/Проекты_Санег/ДЗ_КЗ_дни_просрочки/Исходники/Карточки_Таско/ДЗ+КЗ_Карточки'
path_output = r'/content/drive/MyDrive/Проекты_Санег/ДЗ_КЗ_дни_просрочки/Результаты/Таско_дни_просрочки'


# ***Дата Отчета***

In [3]:
while True:
    date_input = input("Введите дату отчета в формате ДД.ММ.ГГГГ (например, 31.03.2026): ")
    try:
        # Пробуем преобразовать ввод в дату
        report_date = pd.to_datetime(date_input, dayfirst=True)

        print(f"✅ Дата принята: {report_date.strftime('%d %B %Y')}")
        break  # Выходим из цикла, если дата верна
    except Exception:
        print("❌ Ошибка: Неверный формат! Пожалуйста, используйте формат ДД.ММ.ГГГГ")

Введите дату отчета в формате ДД.ММ.ГГГГ (например, 31.03.2026): 31/12/2025
✅ Дата принята: 31 December 2025


# ***Функции для изменения типов***

In [4]:
def change_to_int(df, col_name):
    df.loc[:, col_name] = df[col_name].astype(str).str.replace(',', '.')
    df.loc[:, col_name] = df[col_name].astype(str).str.replace(r'\s+', '', regex=True)
    df.loc[:, col_name]= pd.to_numeric(df[col_name], errors = 'coerce')
    df.loc[:, col_name] = df[col_name].fillna(0).round(2)
    return df
# __________________________________________________________________________________________________
def change_time_type(df, col_name):
    df.loc[:, col_name] = pd.to_datetime(df[col_name], format='%d.%m.%Y', dayfirst=True, errors = 'coerce')
    df = df.dropna(subset=[col_name])
    return df

# __________________________________________________________________________________________________
def process_kartochka(df1, sch_val):

    df1 = df1.copy()

    df1.columns = [f'col_{x + 1}' for x in range(len(df1.columns))]

    df1.loc[:, 'Счет_Загрузки'] = sch_val

    df1 = change_time_type(df1, 'col_1')
    df1 = change_to_int(df1, 'col_6')
    df1 = change_to_int(df1, 'col_8')

    df1 = df1[df1['col_1'] <= report_date]

    debt_mapping = {
        '60': 'Кредиторская задолженность',
        '63': 'Авансы полученные',
        '43': 'Авансы выданные',
        '40': 'Дебиторская задолженность'
    }

    prefix = str(sch_val)[:2]

    df1['Категория_Счета'] = pd.Series([prefix] * len(df1)).map(debt_mapping).fillna('Прочее')

    df2 = df1.copy()

    df1 = df1.loc[df1['col_5'].str[:2] == prefix]
    df1.loc[:, 'Контрагент'] = df1['col_3'].str.split('\r\n').str[0]
    df1.loc[:, 'Контракт']= df1['col_3'].str.split('\r\n').str[1]
    df1 = df1.drop(columns=['col_8'])

    df2 = df2.loc[df2['col_7'].str[:2] == prefix]
    df2.loc[:, 'Контрагент'] = df2['col_4'].str.split('\r\n').str[0]
    df2.loc[:, 'Контракт']= df2['col_4'].str.split('\r\n').str[1]
    df2 = df2.drop(columns=['col_6'])

    df1 = pd.concat([df1, df2], ignore_index=True)

    df1 = df1.drop(columns=['col_3', 'col_4', 'col_2'])
    df1 = df1.rename(columns={'col_1' : 'Дата', 'col_5' : 'Счет_Дт', 'col_6' : 'Сумма_Дт', 'col_7' : 'Счет_Кт', 'col_8' : 'Сумма_Кт'})
    df1 = df1.iloc[:, [0,6,7,1,2,3,8,4,5]]

    df1['Сумма_Дт'] = df1['Сумма_Дт'].fillna(0).round(2)
    df1['Сумма_Кт'] = df1['Сумма_Кт'].fillna(0).round(2)

    grouped = df1.groupby(['Контрагент', 'Контракт', 'Счет_Загрузки'])[['Сумма_Дт', 'Сумма_Кт']].transform('sum')
    mask_balance = grouped['Сумма_Дт'] != grouped['Сумма_Кт']
    df1 = df1[mask_balance].copy()

    return df1


# Обработкa исходников (карточек)

In [5]:
all_frames = []

for file_name in os.listdir(path_input):
    file_path = os.path.join(path_input, file_name)

    # Открываем файл как текст, чтобы определить структуру и найти "Сальдо на начало"
    with open(file_path, 'r', encoding='utf-8') as f:
        lines = f.readlines()

    # Извлекаем sch_val со второй строки, как у вас и было
    second_line = lines[1]
    sch_val = second_line.split()[2]

    # Ищем динамически, на какой строке находится "Сальдо на начало"
    skip_rows_count = None
    for idx, line in enumerate(lines):
        if "Сальдо на начало" in line:
            skip_rows_count = idx
            break

    # Если вдруг в каком-то файле строка не найдена, ставим дефолтное значение 7
    if skip_rows_count is None:
        skip_rows_count = 7

    # Теперь pandas прочитает файл идеально ровно, начиная с нужной строки
    df1 = pd.read_csv(
        file_path,
        sep='\t',
        skiprows=skip_rows_count,
        dtype='str',
        usecols=[0, 1, 2, 3, 4, 5, 7, 8]
    )

    # Обрабатываем и добавляем в общий список
    df1 = process_kartochka(df1, sch_val)
    all_frames.append(df1)

# Объединяем все датафреймы в один
cdf1 = pd.concat(all_frames, ignore_index=True)

/tmp/ipykernel_5870/4252076150.py:5: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df.loc[:, col_name] = df[col_name].fillna(0).round(2)
/tmp/ipykernel_5870/4252076150.py:5: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df.loc[:, col_name] = df[col_name].fillna(0).round(2)
/tmp/ipykernel_5870/4252076150.py:57: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downc

# ***Очистка минусовых остатков***

In [6]:
cdf2 = cdf1.copy()

cdf2['Счет_Загрузки'] = pd.to_numeric(cdf2['Счет_Загрузки'], errors='coerce').astype(int).astype(str)

mask_active = cdf2['Счет_Загрузки'].isin(['4000', '4300'])

cdf2['Поступление'] = np.where(mask_active, cdf2['Сумма_Дт'], cdf2['Сумма_Кт'])
cdf2['Погашение'] = np.where(mask_active, cdf2['Сумма_Кт'], cdf2['Сумма_Дт'])

cdf2 = cdf2.sort_values(by=['Контрагент', 'Контракт', 'Дата'])

cdf2['Чистая_разница'] = cdf2['Поступление'] - cdf2['Погашение']

cdf2['Чистая_разница_4300/4000'] = np.where(cdf2['Счет_Загрузки'].isin(['4000', '4300']), cdf2['Чистая_разница'], 0)
cdf2['Чистая_разница_6300/6000'] = np.where(~cdf2['Счет_Загрузки'].isin(['4000', '4300']), cdf2['Чистая_разница'], 0)

total_sum_4300_4000 = cdf2.groupby(['Контрагент', 'Контракт'])['Чистая_разница_4300/4000'].transform('sum')
total_sum_6300_6000 = cdf2.groupby(['Контрагент', 'Контракт'])['Чистая_разница_6300/6000'].transform('sum')

cdf2['4300/4000'] = np.where(cdf2['Счет_Загрузки'].isin(['4000', '4300']), total_sum_4300_4000, 0)
cdf2['6300/6000'] = np.where(~cdf2['Счет_Загрузки'].isin(['4000', '4300']), total_sum_6300_6000, 0)

bad_vals = [0, -0.01, 0.01, -0.00, 0.00]

cdf2 = cdf2.loc[
    ~cdf2['4300/4000'].isin(bad_vals) |
    ~cdf2['6300/6000'].isin(bad_vals)
]

cdf2['Поступ_изм_40/43'] = np.where(cdf2['Счет_Загрузки'].isin(['4000', '4300']), np.where(cdf2['4300/4000'] < 0, cdf2['Погашение'], cdf2['Поступление']), 0)
cdf2['Погаш_изм_40/43'] = np.where(cdf2['Счет_Загрузки'].isin(['4000', '4300']), np.where(cdf2['4300/4000'] < 0, cdf2['Поступление'], cdf2['Погашение']), 0)

cdf2['Поступ_изм_60/63'] = np.where(~cdf2['Счет_Загрузки'].isin(['4000', '4300']), np.where(cdf2['6300/6000'] < 0 , cdf2['Погашение'], cdf2['Поступление']), 0)
cdf2['Погаш_изм_60/63'] = np.where(~cdf2['Счет_Загрузки'].isin(['4000', '4300']), np.where(cdf2['6300/6000'] < 0 , cdf2['Поступление'], cdf2['Погашение']), 0 )

cdf2['Поступ_изм'] = cdf2['Поступ_изм_40/43'] + cdf2['Поступ_изм_60/63']
cdf2['Погаш_изм'] = cdf2['Погаш_изм_40/43'] + cdf2['Погаш_изм_60/63']

cdf2 = cdf2.drop(columns=['Поступ_изм_40/43', 'Погаш_изм_40/43', 'Поступ_изм_60/63', 'Погаш_изм_60/63'])

cdf2['Категория_Счета_нов'] = np.where(cdf2['4300/4000'] < 0, np.where(cdf2['Счет_Загрузки'] == '4000', 'Авансы полученные', 'Кредиторская задолженность') , cdf2['Категория_Счета'])
cdf2['Категория_Счета_нов'] = np.where(cdf2['6300/6000'] < 0, np.where(cdf2['Счет_Загрузки'] == '6000', 'Авансы выданные', 'Дебиторская задолженность'), cdf2['Категория_Счета_нов'])

df_minus_ost = cdf2.copy()

cdf2 = cdf2.drop(columns=['Поступление','Погашение', 'Чистая_разница', 'Чистая_разница_4300/4000', 'Чистая_разница_6300/6000', '4300/4000', '6300/6000'])


# ***Ручное перекрытие остатков между счетов***

In [7]:
cdf2['Чистый_остаток_40/43'] = np.where(cdf2['Счет_Загрузки'].isin(['4000', '4300']) ,cdf2['Поступ_изм'] - cdf2['Погаш_изм'], 0)
cdf2['Чистый_остаток_60/63'] = np.where(~cdf2['Счет_Загрузки'].isin(['4000', '4300']) ,cdf2['Поступ_изм'] - cdf2['Погаш_изм'], 0)

cdf2['Группа_40/43'] =  cdf2.groupby(['Контрагент', 'Контракт'])['Чистый_остаток_40/43'].transform('sum')
cdf2['Группа_60/63'] =  cdf2.groupby(['Контрагент', 'Контракт'])['Чистый_остаток_60/63'].transform('sum')

cdf2 = cdf2.drop(columns=['Чистый_остаток_40/43', 'Чистый_остаток_60/63'])

cdf2['Поступ_40/43'] = np.where(cdf2['Счет_Загрузки'].isin(['4000', '4300']) ,cdf2['Поступ_изм'], 0)
cdf2['Погаш_40/43'] = np.where(cdf2['Счет_Загрузки'].isin(['4000', '4300']) ,cdf2['Погаш_изм'], 0)

cdf2['Поступ_60/63'] = np.where(~cdf2['Счет_Загрузки'].isin(['4000', '4300']) ,cdf2['Поступ_изм'], 0)
cdf2['Погаш_60/63'] = np.where(~cdf2['Счет_Загрузки'].isin(['4000', '4300']) ,cdf2['Погаш_изм'], 0)

cdf2['Статус_перекр'] = np.where((cdf2['Счет_Загрузки'].isin(['4000', '4300'])) & (cdf2['Группа_40/43'] < cdf2['Группа_60/63'])  & (cdf2['Группа_40/43'] > 1) , 'Ручное переркытие', None)
cdf2['Статус_перекр'] = np.where((~cdf2['Счет_Загрузки'].isin(['4000', '4300'])) & (cdf2['Группа_60/63'] < cdf2['Группа_40/43'])  & (cdf2['Группа_60/63'] > 1) , 'Ручное переркытие', cdf2['Статус_перекр'])

cdf2['Поступ_43/40_испр'] = np.where(cdf2['Статус_перекр'] == 'Ручное переркытие', cdf2['Погаш_40/43'], cdf2['Поступ_40/43'])
cdf2['Погаш_43/40_испр'] = np.where(cdf2['Статус_перекр'] == 'Ручное переркытие', cdf2['Поступ_40/43'], cdf2['Погаш_40/43'])

cdf2['Поступ_63/60_испр'] = np.where(cdf2['Статус_перекр'] == 'Ручное переркытие', cdf2['Погаш_60/63'], cdf2['Поступ_60/63'])
cdf2['Погаш_63/60_испр'] = np.where(cdf2['Статус_перекр'] == 'Ручное переркытие', cdf2['Поступ_60/63'], cdf2['Погаш_60/63'])

cdf2['Поступ_фин'] = cdf2['Поступ_43/40_испр'] + cdf2['Поступ_63/60_испр']
cdf2['Погаш_фин'] = cdf2['Погаш_43/40_испр'] + cdf2['Погаш_63/60_испр']

cdf2 = cdf2.drop(columns=['Поступ_40/43', 'Погаш_40/43', 'Поступ_60/63', 'Погаш_60/63', 'Поступ_43/40_испр', 'Погаш_43/40_испр', 'Поступ_63/60_испр', 'Погаш_63/60_испр'])

cdf2['Категория_Счета_нов'] = np.where((cdf2['Категория_Счета_нов'] != cdf2['Категория_Счета']) & (cdf2['Группа_40/43'] > cdf2['Группа_60/63']), cdf2['Категория_Счета'], cdf2['Категория_Счета_нов'])

cdf2['Кат_счетов_испр'] = np.where((cdf2['Счет_Загрузки'] == '4000') & (cdf2['Статус_перекр'] == 'Ручное переркытие'), 'Авансы полученные',
                                   np.where((cdf2['Счет_Загрузки'] == '4300') & (cdf2['Статус_перекр'] == 'Ручное переркытие'), 'Кредиторская задолженность',
                                   np.where((cdf2['Счет_Загрузки'] == '6000') & (cdf2['Статус_перекр'] == 'Ручное переркытие'), 'Авансы выданные',
                                   np.where((cdf2['Счет_Загрузки'] == '6300') & (cdf2['Статус_перекр'] == 'Ручное переркытие'), 'Дебиторская задолженность', cdf2['Категория_Счета_нов']))))

cdf2['Сумма_Критерия'] = (
    cdf2.groupby(['Контрагент', 'Контракт', 'Кат_счетов_испр'])['Поступ_фин'].transform('sum') -
    cdf2.groupby(['Контрагент', 'Контракт', 'Кат_счетов_испр'])['Погаш_фин'].transform('sum')
)

cdf2['Кат_счетов_испр'] = np.where(cdf2['Сумма_Критерия'] < 0, cdf2['Категория_Счета_нов'], cdf2['Кат_счетов_испр'])

cdf2['Кат_счетов_испр'] = np.where((cdf2['Сумма_Критерия'] < 0) & (cdf2['Статус_перекр'] ==  'Ручное переркытие') & (cdf2['Кат_счетов_испр'] == 'Авансы полученные'), 'Авансы выданные', cdf2['Кат_счетов_испр'])

df_perekritie = cdf2.copy()

cdf2 = cdf2.drop(columns=['Поступ_изм', 'Погаш_изм', 'Категория_Счета_нов', 'Группа_40/43', 'Группа_60/63', 'Сумма_Критерия', 'Статус_перекр'])

cdf2['Разница_упр'] = cdf2['Поступ_фин'] - cdf2['Погаш_фин']

# **Минусовые остатки ДФ для анализа**

In [8]:
df_minus_ost = df_minus_ost.loc[df_minus_ost['Категория_Счета'] != df_minus_ost['Категория_Счета_нов']]

df_minus_ost = df_minus_ost.groupby(['Контрагент', 'Контракт', 'Счет_Загрузки', 'Категория_Счета' ], as_index=False)['Чистая_разница'].sum()

df_minus_ost['Статус_Мин'] = 'Минусовые_остатки'

df_minus_ost = df_minus_ost.loc[df_minus_ost['Чистая_разница'] < -1]

cdf3 = cdf2.copy()

cdf3 = cdf3.groupby(['Контрагент', 'Контракт', 'Счет_Загрузки', 'Категория_Счета' ], as_index=False)['Разница_упр'].sum()

df_minus_ost = df_minus_ost.merge(cdf3[['Контрагент', 'Контракт', 'Счет_Загрузки', 'Категория_Счета', 'Разница_упр']], on=['Контрагент', 'Контракт', 'Счет_Загрузки', 'Категория_Счета'], how='left')

df_minus_ost['Статус_ошибка'] = np.where((df_minus_ost['Разница_упр'] + df_minus_ost['Чистая_разница']) != 0 , 'Надо_проверить', 'Нормально')


# Ручное перекрытие ДФ для анализа

In [9]:
df_perekritie = df_perekritie.loc[df_perekritie['Статус_перекр'] == 'Ручное переркытие', ['Дата', 'Контрагент', 'Контракт', 'Группа_40/43', 'Группа_60/63', 'Статус_перекр', 'Счет_Загрузки' ]]

df_perekritie = df_perekritie.drop_duplicates(subset=['Контрагент', 'Контракт'], keep='first')

df_perekritie = df_perekritie.rename(columns={'Группа_40/43' : 'Остатки_на_счетах_4000_4300', 'Группа_60/63' : 'Остатки_на_счетах_6000_6300'})

df_perekritie['Сумма_переноса_из_4000/4300'] = np.where(df_perekritie['Остатки_на_счетах_4000_4300'] > df_perekritie['Остатки_на_счетах_6000_6300'], 0, df_perekritie['Остатки_на_счетах_4000_4300'])

df_perekritie['Сумма_переноса_из_6000/6300'] = np.where(df_perekritie['Остатки_на_счетах_4000_4300'] > df_perekritie['Остатки_на_счетах_6000_6300'], df_perekritie['Остатки_на_счетах_6000_6300'], 0)

# ***Дополнительное исправление минусовых остатков***

In [10]:
keys_df = df_minus_ost[['Контрагент', 'Контракт', 'Счет_Загрузки', 'Статус_ошибка', 'Статус_Мин']].copy()

cdf2 = cdf2.merge(keys_df, on=['Контрагент', 'Контракт', 'Счет_Загрузки'], how='left')
cdf2['Статус_Мин'] = cdf2['Статус_Мин'].fillna('Нет')
cdf2['Статус_ошибка'] = cdf2['Статус_ошибка'].fillna('Нет')

cdf2['Поступ_фин'] = np.where(cdf2['Статус_ошибка'] == 'Надо_проверить', cdf2['Сумма_Дт'], cdf2['Поступ_фин'])
cdf2['Погаш_фин'] = np.where(cdf2['Статус_ошибка'] == 'Надо_проверить', cdf2['Сумма_Кт'], cdf2['Погаш_фин'])

df_minus_ost = df_minus_ost.drop(columns=['Разница_упр','Статус_ошибка'])
cdf2 = cdf2.drop(columns=['Разница_упр', 'Статус_ошибка', 'Статус_Мин'])

cdf2['Разн_упр_2'] = cdf2['Поступ_фин'] - cdf2['Погаш_фин']
cdf4 = cdf2.copy()
cdf4 = cdf4.groupby(['Контрагент', 'Контракт', 'Счет_Загрузки', 'Категория_Счета' ], as_index=False)['Разн_упр_2'].sum()

df_minus_ost = df_minus_ost.merge(cdf4[['Контрагент', 'Контракт', 'Счет_Загрузки', 'Категория_Счета', 'Разн_упр_2']], on=['Контрагент', 'Контракт', 'Счет_Загрузки', 'Категория_Счета'], how='left')
df_minus_ost['Статус_ошибка_2'] = np.where((df_minus_ost['Разн_упр_2'] + df_minus_ost['Чистая_разница']) != 0 , 'Надо_проверить', 'Нормально')


keys_df_2 = df_minus_ost[['Контрагент', 'Контракт', 'Счет_Загрузки', 'Статус_ошибка_2']].copy()
cdf2 = cdf2.merge(keys_df_2, on = ['Контрагент', 'Контракт', 'Счет_Загрузки'], how='left')

cdf2['Поступ_фин'] = np.where(cdf2['Статус_ошибка_2'] == 'Надо_проверить', cdf2['Сумма_Кт'], cdf2['Поступ_фин'])
cdf2['Погаш_фин'] = np.where(cdf2['Статус_ошибка_2'] == 'Надо_проверить', cdf2['Сумма_Дт'], cdf2['Погаш_фин'])

df_minus_ost = df_minus_ost.drop(columns=['Разн_упр_2', 'Статус_ошибка_2', 'Статус_Мин'])
cdf2 = cdf2.drop(columns=['Статус_ошибка_2', 'Разн_упр_2'])
cdf2['Разница_1С'] = np.where(cdf2['Счет_Загрузки'].isin(['4000', '4300']), cdf2['Сумма_Дт'] - cdf2['Сумма_Кт'],cdf2['Сумма_Кт'] - cdf2['Сумма_Дт'])
cdf2['Разница_упр'] = cdf2['Поступ_фин'] - cdf2['Погаш_фин']

# ***Дополнительное исправление Ручного перекрытия остатков между счетов***


In [11]:
cdf3 = cdf2.copy()

cdf3 = cdf3.pivot_table(index=['Контрагент', 'Контракт'], columns="Кат_счетов_испр", values="Разница_упр", aggfunc="sum", fill_value=0).reset_index()

cdf3['Статус_перекр'] = np.where( ((cdf3['Авансы выданные'] == cdf3['Кредиторская задолженность']) & (cdf3['Авансы выданные'] > 0 ))
| ((cdf3['Авансы полученные'] == cdf3['Дебиторская задолженность']) & (cdf3['Авансы полученные'] > 0)), 'Ручное переркытие', None)

cdf3 = cdf3.loc[cdf3['Статус_перекр'] == 'Ручное переркытие']

mask = cdf2["Кат_счетов_испр"].isin(["Авансы выданные", "Дебиторская задолженность"])
merged_part = cdf2[mask].merge(cdf3[["Контрагент", "Контракт", "Статус_перекр"]], on=["Контрагент", "Контракт"], how="left")
cdf2.loc[mask, "Статус_перекр"] = merged_part["Статус_перекр"].values

cdf2['Поступ_фин'] = np.where( (cdf2['Статус_перекр'] == 'Ручное переркытие') & (cdf2['Кат_счетов_испр'] == 'Авансы выданные') , cdf2['Сумма_Кт'], cdf2['Поступ_фин'])
cdf2['Погаш_фин'] = np.where( (cdf2['Статус_перекр'] == 'Ручное переркытие') & (cdf2['Кат_счетов_испр'] == 'Авансы выданные') , cdf2['Сумма_Дт'], cdf2['Погаш_фин'])

cdf2['Поступ_фин'] = np.where( (cdf2['Статус_перекр'] == 'Ручное переркытие') & (cdf2['Кат_счетов_испр'] == 'Дебиторская задолженность') , cdf2['Сумма_Кт'], cdf2['Поступ_фин'])
cdf2['Погаш_фин'] = np.where( (cdf2['Статус_перекр'] == 'Ручное переркытие') & (cdf2['Кат_счетов_испр'] == 'Дебиторская задолженность') , cdf2['Сумма_Дт'], cdf2['Погаш_фин'])

cdf2['Кат_счетов_испр'] = np.where((cdf2['Статус_перекр'] == 'Ручное переркытие') & (cdf2['Кат_счетов_испр'] == 'Авансы выданные'), 'Кредиторская задолженность', cdf2['Кат_счетов_испр'])
cdf2['Кат_счетов_испр'] = np.where((cdf2['Статус_перекр'] == 'Ручное переркытие') & (cdf2['Кат_счетов_испр'] == 'Дебиторская задолженность'), 'Авансы полученные', cdf2['Кат_счетов_испр'])

cdf2['Раз_упр_НОВАЯ'] = cdf2['Поступ_фин'] - cdf2['Погаш_фин']

cdf2 = cdf2.drop(columns=['Статус_перекр', 'Разница_упр'])

df_perekritie = df_perekritie.drop(columns=['Дата', 'Счет_Загрузки'])

cdf3['Остатки_на_счетах_4000_4300'] = cdf3['Авансы выданные'] + cdf3['Дебиторская задолженность']
cdf3['Остатки_на_счетах_6000_6300'] = cdf3['Авансы полученные'] + cdf3['Кредиторская задолженность']
cdf3['Сумма_переноса_из_4000/4300'] = cdf3['Остатки_на_счетах_4000_4300']
cdf3['Сумма_переноса_из_6000/6300'] = cdf3['Остатки_на_счетах_6000_6300']
cdf3 = cdf3.drop(columns=['Авансы выданные', 'Авансы полученные', 'Кредиторская задолженность', 'Дебиторская задолженность'])
df_perekritie = pd.concat([df_perekritie, cdf3], ignore_index=True)

# ***Расчет дней просрочек***

In [12]:
cdf2 = cdf2.sort_values(['Контрагент', 'Контракт', 'Дата','Кат_счетов_испр'])
group_totals = cdf2.groupby(['Контрагент', 'Контракт'])
cdf2['Общая_Оплата'] =  group_totals['Погаш_фин'].transform('sum').round(2)
cdf2['Кум_Поступ'] =  group_totals['Поступ_фин'].cumsum().round(2)
cdf2['Общая_Поступл'] =  group_totals['Поступ_фин'].transform('sum').round(2)

cdf2['Невыпл_сумма_строк'] = (cdf2['Кум_Поступ'] - cdf2['Общая_Оплата']).round(2)

cdf2['Сумма_долга'] = np.where(cdf2['Поступ_фин'] > 0, cdf2['Невыпл_сумма_строк'].clip(lower=0), 0.0)

mask_positive = cdf2['Поступ_фин'] > 0
cdf2.loc[mask_positive, 'Сумма_долга'] = cdf2[['Сумма_долга', 'Поступ_фин']].min(axis=1)

cdf2['is_last_row'] = cdf2.groupby(['Контрагент', 'Контракт']).cumcount(ascending=False) == 0


group_sums = cdf2.groupby(['Контрагент', 'Контракт'])['Сумма_долга'].transform('sum')

cdf2.loc[cdf2['is_last_row'] == True, 'Сумма_долга'] = (
    cdf2['Общая_Поступл'] - cdf2['Общая_Оплата'] -
    (group_sums - cdf2['Сумма_долга'])
).round(2)

cdf2['Дата'] = pd.to_datetime(cdf2['Дата'], dayfirst=True)
cdf2['Дней_просрочки'] = (report_date - cdf2['Дата']).dt.days

cdf2.loc[cdf2['Сумма_долга'] < 0, 'Дней_просрочки'] = -1

# 6. Распределение по бакетам
bins = [-np.inf, 0, 30, 60, 90, 120, 150, np.inf]
labels = ['Не просрочено', '0-30 дней', '30-60 дней', '60-90 дней', '90-120 дней', '120-150 дней', '150+ дней']

cdf2['Группа_просрочек'] = pd.cut(cdf2['Дней_просрочки'], bins=bins, labels=labels)

cdf2.drop(columns=['Невыпл_сумма_строк', 'is_last_row', 'Общая_Поступл','Общая_Оплата', 'Кум_Поступ'], inplace=True, errors='ignore')

cdf2 = cdf2.rename(columns={'Сумма_долга' : 'Сумма_остаток', 'Дней_просрочки': 'Дни_просрочек'})

# ***Дополнительное исправление Категории Счетов и других ошибок***

In [13]:
df_kat = cdf2.copy()

df_kat = df_kat.groupby(['Контрагент', 'Контракт', 'Кат_счетов_испр'], as_index=False)[['Сумма_остаток', 'Раз_упр_НОВАЯ']].sum()
df_kat['Ошибка_Кат'] = np.where(((df_kat['Сумма_остаток'] - df_kat['Раз_упр_НОВАЯ'])) > 0 , 'Ошибка_Кат', None)
df_kat = df_kat.loc[df_kat['Ошибка_Кат'] == 'Ошибка_Кат']

cdf2 = cdf2.merge(df_kat[['Контрагент', 'Контракт', 'Кат_счетов_испр', 'Ошибка_Кат']], on=['Контрагент', 'Контракт', 'Кат_счетов_испр'], how='left')

mapping_dict = {
    "Авансы полученные": "Дебиторская задолженность",
    "Авансы выданные": "Кредиторская задолженность",
    "Кредиторская задолженность": "Авансы выданные",
    "Дебиторская задолженность": "Авансы полученные",
}

cdf2["Кат_НОВАЯ"] = np.where(
    cdf2["Ошибка_Кат"] == "Ошибка_Кат",
    cdf2["Кат_счетов_испр"].map(mapping_dict),  # Сам ВПР происходит здесь
    cdf2["Кат_счетов_испр"],  # Если ошибки нет, оставляем как было
)

cdf2 = cdf2.drop(columns=['Ошибка_Кат', 'Кат_счетов_испр'])

# ***Порядок в названиях таблиц***

In [14]:
cdf2 = cdf2.rename(columns={'Сумма_Дт' : 'Поступ_НСБУ','Сумма_Кт' : 'Выб_НСБУ', 'Категория_Счета' : 'Категория_НСБУ', 'Погаш_фин' : 'Выб_Управ','Поступ_фин' : 'Поступ_Управ',
                          'Разница_1С' : 'Сумма_НСБУ', 'Раз_упр_НОВАЯ' : 'Сумма_Управ', 'Сумма_остаток' : 'Сумма_Остаток', 'Кат_НОВАЯ' : 'Категория_Управ' })

df_minus_ost = df_minus_ost.rename(columns={'Чистая_разница' : 'Минусовой_Остаток','Категория_Счета' : 'Категория_НСБУ'})

cdf2['Дата'] = cdf2['Дата'].dt.date


# ***Сводные таблицы для диграмм***

In [15]:
PV_NSBU = cdf2.groupby('Категория_НСБУ', as_index=False)['Сумма_НСБУ'].sum()
PV_Min_ost = df_minus_ost.groupby('Категория_НСБУ', as_index=False)['Минусовой_Остаток'].sum()
PV_Perekritie = pd.pivot_table(df_perekritie, values=['Сумма_переноса_из_4000/4300', 'Сумма_переноса_из_6000/6300'], index=['Статус_перекр'], aggfunc='sum')
PV_Uprav = cdf2.groupby('Категория_Управ', as_index=False)['Сумма_Управ'].sum()

# ____________________________________________________Дебиторская Задолженность____________________________________________________

PV_DZ = cdf2[cdf2['Категория_Управ'] == 'Дебиторская задолженность'].groupby('Группа_просрочек', as_index=False, observed=True)['Сумма_Остаток'].sum()
PV_DZ_DNI = pd.pivot_table(cdf2[cdf2['Категория_Управ'] == 'Дебиторская задолженность'],
                           values='Сумма_Остаток', index=['Контрагент', 'Контракт'], columns= 'Группа_просрочек', aggfunc='sum', fill_value=0, observed=True).reset_index()
PV_DZ_DNI['Общий_итог']= PV_DZ_DNI.iloc[:, 2:].sum(axis = 1)
PV_DZ_DNI = PV_DZ_DNI.sort_values(by = 'Общий_итог', ascending=False)
PV_DZ_DNI = PV_DZ_DNI.loc[PV_DZ_DNI['Общий_итог'] != 0]

# ____________________________________________________Авансы выданные____________________________________________________

PV_Avan_43= cdf2[cdf2['Категория_Управ'] == 'Авансы выданные'].groupby('Группа_просрочек', as_index=False, observed=True)['Сумма_Остаток'].sum()
PV_Avanc_DNI_43 = pd.pivot_table(cdf2[cdf2['Категория_Управ'] == 'Авансы выданные'],
                           values='Сумма_Остаток', index=['Контрагент', 'Контракт'], columns= 'Группа_просрочек', aggfunc='sum', fill_value=0, observed=True).reset_index()
PV_Avanc_DNI_43['Общий_итог']= PV_Avanc_DNI_43.iloc[:, 2:].sum(axis = 1)
PV_Avanc_DNI_43= PV_Avanc_DNI_43.sort_values(by = 'Общий_итог', ascending=False)
PV_Avanc_DNI_43 = PV_Avanc_DNI_43.loc[PV_Avanc_DNI_43['Общий_итог'] != 0]

# ____________________________________________________Кредиторская задолженность____________________________________________________

PV_KZ = cdf2[cdf2['Категория_Управ'] == 'Кредиторская задолженность'].groupby('Группа_просрочек', as_index=False, observed=True)['Сумма_Остаток'].sum()
PV_KZ_DNI = pd.pivot_table(cdf2[cdf2['Категория_Управ'] == 'Кредиторская задолженность'],
                           values='Сумма_Остаток', index=['Контрагент', 'Контракт'], columns= 'Группа_просрочек', aggfunc='sum', fill_value=0, observed=True).reset_index()
PV_KZ_DNI['Общий_итог']= PV_KZ_DNI.iloc[:, 2:].sum(axis = 1)
PV_KZ_DNI = PV_KZ_DNI.sort_values(by = 'Общий_итог', ascending=False)
PV_KZ_DNI = PV_KZ_DNI.loc[PV_KZ_DNI['Общий_итог'] != 0]

# ____________________________________________________Авансы полученные____________________________________________________________

PV_Avan_63 = cdf2[cdf2['Категория_Управ'] == 'Авансы полученные'].groupby('Группа_просрочек', as_index=False, observed=True)['Сумма_Остаток'].sum()
PV_Avan_DNI_63 = pd.pivot_table(cdf2[cdf2['Категория_Управ'] == 'Авансы полученные'],
                           values='Сумма_Остаток', index=['Контрагент', 'Контракт'], columns= 'Группа_просрочек', aggfunc='sum', fill_value=0, observed=True).reset_index()
PV_Avan_DNI_63['Общий_итог']= PV_Avan_DNI_63.iloc[:, 2:].sum(axis = 1)
PV_Avan_DNI_63 = PV_Avan_DNI_63.sort_values(by = 'Общий_итог', ascending=False)
PV_Avan_DNI_63 = PV_Avan_DNI_63.loc[PV_Avan_DNI_63['Общий_итог'] != 0]

# ____________________________________________________Авансы выданные/Кредиторская задолженность____________________________________________________

Df_avans_kz = cdf2[cdf2['Категория_Управ'].isin(['Авансы выданные', 'Кредиторская задолженность'])]
Df_avans_kz = Df_avans_kz.loc[:,['Контрагент', 'Контракт', 'Группа_просрочек', 'Сумма_Остаток', 'Категория_Управ']]
Df_avans_kz['Сумма_Остаток'] = np.where(Df_avans_kz['Категория_Управ'] == 'Кредиторская задолженность', Df_avans_kz['Сумма_Остаток'] * -1, Df_avans_kz['Сумма_Остаток'])
Df_avans_kz_gr = Df_avans_kz.groupby('Группа_просрочек', as_index=False, observed=True)['Сумма_Остаток'].sum()
Df_avans_kz_DNI = pd.pivot_table(Df_avans_kz, values='Сумма_Остаток', index=['Контрагент', 'Контракт'], columns= 'Группа_просрочек', aggfunc='sum', fill_value=0, observed=True).reset_index()
Df_avans_kz_DNI['Общий_итог']= Df_avans_kz_DNI.iloc[:, 2:].sum(axis = 1)
Df_avans_kz_DNI = Df_avans_kz_DNI.sort_values(by = 'Общий_итог', ascending=False)
Df_avans_kz_DNI = Df_avans_kz_DNI.loc[Df_avans_kz_DNI['Общий_итог'] != 0]

# ____________________________________________________Дебиторская Задолженность/Авансы полученные____________________________________________________

Df_debet_avans = cdf2[cdf2['Категория_Управ'].isin(['Дебиторская задолженность', 'Авансы полученные'])]
Df_debet_avans = Df_debet_avans.loc[:,['Контрагент', 'Контракт', 'Группа_просрочек', 'Сумма_Остаток', 'Категория_Управ']]
Df_debet_avans['Сумма_Остаток'] = np.where(Df_debet_avans['Категория_Управ'] == 'Авансы полученные', Df_debet_avans['Сумма_Остаток'] * -1, Df_debet_avans['Сумма_Остаток'])
Df_debet_avans_gr = Df_debet_avans.groupby('Группа_просрочек', as_index=False, observed=True)['Сумма_Остаток'].sum()
Df_debet_avans_DNI = pd.pivot_table(Df_debet_avans, values='Сумма_Остаток', index=['Контрагент', 'Контракт'], columns= 'Группа_просрочек', aggfunc='sum', fill_value=0, observed=True).reset_index()
Df_debet_avans_DNI['Общий_итог']= Df_debet_avans_DNI.iloc[:, 2:].sum(axis = 1)
Df_debet_avans_DNI = Df_debet_avans_DNI.sort_values(by = 'Общий_итог', ascending=False)
Df_debet_avans_DNI = Df_debet_avans_DNI.loc[Df_debet_avans_DNI['Общий_итог'] != 0]


# ***Функции для изменения стиля таблицы***

In [16]:
def apply_sheet_style(ws, df, col_widths, header_color="1F497D"):
    """Универсальная функция для быстрой стилизации любого листа openpyxl."""
    # 1. Стили для шапки и ячеек
    fill = PatternFill(
        start_color=header_color, end_color=header_color, fill_type="solid"
    )
    font_h = Font(name="Segoe UI", size=11, bold=True, color="FFFFFF")
    font_r = Font(name="Segoe UI", size=10)
    border = Border(
        left=Side(style="thin", color="D9D9D9"),
        right=Side(style="thin", color="D9D9D9"),
        top=Side(style="thin", color="D9D9D9"),
        bottom=Side(style="thin", color="D9D9D9"),
    )

    # 2. Быстрое форматирование шапки
    ws.row_dimensions[1].height = 26
    for cell in ws[1]:
        cell.fill, cell.font, cell.border = fill, font_h, border
        cell.alignment = Alignment(
            horizontal="center", vertical="center", wrap_text=True
        )

    # 3. Форматирование строк данных (числа / текст)
    for row in ws.iter_rows(
        min_row=2, max_row=ws.max_row, min_col=1, max_col=df.shape[1]
    ):
        for cell in row:
            cell.font, cell.border = font_r, border
            if isinstance(cell.value, (int, float)):
                cell.number_format = "#,##0.00"
                cell.alignment = Alignment(horizontal="right", vertical="center")
            else:
                cell.alignment = Alignment(horizontal="left", vertical="center")

    # 4. Применяем переданную ширину столбцов
    for col_letter, width in col_widths.items():
        ws.column_dimensions[col_letter].width = width

widths_minus = {"A": 40, "B": 40, "C": 15, "D": 30, "E": 25}
widths_perekritie = {"A": 40, "B": 40, "C": 20, "D": 20, "E": 20, "F": 25, "G": 25}
widths_svod = {"A": 10, "B": 35, "C": 35, "D": 10, "E": 17, "F": 10, "G": 17, "H": 15, "I": 20, "J": 17, "K": 17, "L": 17, "M": 17, "N": 17, "O": 15, "P": 17, "Q": 17}

# ***Сохранение файлов***

In [17]:
final_path = os.path.join(path_output, f'Таско_дни_просрочки_на_{report_date}.xlsx')

with pd.ExcelWriter(final_path) as writer:
    cdf2.to_excel(writer, sheet_name ='Свод', index=False)
    df_minus_ost.to_excel(writer, sheet_name ='Минусовые_Остатки', index=False)
    df_perekritie.to_excel(writer, sheet_name ='Ручное_перекрытие', index=False)

    PV_NSBU.to_excel(writer, sheet_name ='PV_NSBU', index=False)
    PV_Min_ost.to_excel(writer, sheet_name ='PV_Min_ost', index=False)
    PV_Perekritie.to_excel(writer, sheet_name ='PV_Perekritie', index=False)
    PV_Uprav.to_excel(writer, sheet_name ='PV_Uprav', index=False)

    PV_DZ.to_excel(writer, sheet_name ='PV_DZ', index=False)
    PV_DZ_DNI.to_excel(writer, sheet_name ='PV_DZ_DNI', index=False)
    PV_Avan_43.to_excel(writer, sheet_name ='PV_Avan_43', index=False)
    PV_Avanc_DNI_43.to_excel(writer, sheet_name ='PV_Avanc_DNI_43', index=False)
    PV_KZ.to_excel(writer, sheet_name ='PV_KZ', index=False)
    PV_KZ_DNI.to_excel(writer, sheet_name ='PV_KZ_DNI', index=False)
    PV_Avan_63.to_excel(writer, sheet_name ='PV_Avan_63', index=False)
    PV_Avan_DNI_63.to_excel(writer, sheet_name ='PV_Avan_DNI_63', index=False)

    Df_avans_kz_gr.to_excel(writer, sheet_name ='Df_avans_kz', index=False)
    Df_avans_kz_DNI.to_excel(writer, sheet_name ='Df_avans_kz_DNI', index=False)

    Df_debet_avans_gr.to_excel(writer, sheet_name ='Df_debet_avans', index=False)
    Df_debet_avans_DNI.to_excel(writer, sheet_name ='Df_debet_avans_DNI', index=False)

    # apply_sheet_style(writer.sheets["Свод"], cdf2, widths_svod)
    # apply_sheet_style(
    #     writer.sheets["Минусовые_Остатки"], df_minus_ost, widths_minus
    # )
    # apply_sheet_style(
    #     writer.sheets["Ручное_перекрытие"],
    #     df_perekritie,
    #     widths_perekritie,
    #     header_color="2F5597",
    # )